# Cyclopropene in its ground and excited states

These snapshots use the same SA2-CASSCF(4,3)/cc-pVDZ geometry for S₀ and S₁. The
analytic gradients and electric dipoles come from the accompanying forte2
fixture. Run the cells in order. When a cell says a result is ready, run the
next cell to read it. The kernel processes browser replies between cells.

Install the widget with `pip install -e python` from the repository root and
serve that directory with `python3 -m http.server 8000 --bind 127.0.0.1`. Set
`VIBEMOL_VIEWER_URL` to select another VibeMol server.

In [ ]:
import json
from math import hypot
import os
from pathlib import Path

from IPython.display import display
from vibemol import Viewer

fixtures = Path("../tests/data")
properties = json.loads((fixtures / "cyclopropene_sa2cas43_properties.json").read_text())
viewer = Viewer(url=os.environ.get("VIBEMOL_VIEWER_URL"), source="local")
display(viewer)

In [ ]:
def request_series(label, *requests):
    status = display(f"Waiting for {label}", display_id=True)
    results = []
    pending = iter(requests)

    def advance(future=None):
        try:
            if future is not None:
                results.append(future.result())
            request = next(pending, None)
            if request is None:
                status.update(f"{label} ready")
            else:
                request().add_done_callback(advance)
        except Exception as error:
            status.update(f"{label} failed: {error}")

    advance()
    return results


molden = request_series(
    "Molden orbitals",
    lambda: viewer.load(fixtures / "cyclopropene_sa2cas43.molden"),
    viewer.orbitals,
)

In [ ]:
active = [row for row in molden[-1]["orbitals"] if row["class"] == "active"]
print("Active occupations:", ", ".join(f'{row["occupation"]:.3f}' for row in active))
surface = request_series(
    "View and orbital",
    lambda: viewer.set_mode("view"),
    lambda: viewer.show_orbitals([active[1]["index"]]),
    lambda: viewer.set_view(axis="+x", projection="orthographic"),
)

## Forces on the same scale

The arrows show forces, the negative of the analytic energy gradient. Scale the
longest S₁ arrow to 1.0 Å and keep that scale for S₀. The quieter ground-state
panel reflects the smaller forces at this geometry.

In [ ]:
print("Presented orbital", surface[1]["visibleIndices"][0], "in View mode")
force = {state: [[-component for component in atom] for atom in gradient]
         for state, gradient in properties["gradientEhPerBohr"].items()}
atoms = properties["atomicNumbers"]
capture = lambda: viewer.snapshot(width=640, height=460)
s1_force = request_series(
    "S1 forces",
    viewer.hide_orbitals,
    lambda: viewer.set_atom_vectors("forces", force["S1"], "#B22D73", max_length=1.0, atomic_numbers=atoms),
    capture,
)

In [ ]:
force_scale = s1_force[1]["sets"][0]["scale"]
print(f"S1 force scale: {force_scale:.6f} Å per (Eh/bohr)")
display(s1_force[-1])
s0_force = request_series(
    "S0 forces",
    lambda: viewer.set_atom_vectors("forces", force["S0"], "#B22D73", scale=force_scale, atomic_numbers=atoms),
    capture,
)

In [ ]:
s0_applied_scale = s0_force[0]["sets"][0]["scale"]
print(f"S0 uses the S1 force scale: {s0_applied_scale == force_scale}")
print(f'Longest S0 arrow: {force_scale * max(hypot(*atom) for atom in force["S0"]):.3f} Å')
display(s0_force[-1])

## Electric dipole reversal

Both dipoles are anchored at the center of mass and use the same
Å-per-atomic-unit display scale. The total dipole changes direction along z
from S₀ to S₁.

In [ ]:
dipoles = properties["dipoleAu"]
dipole_scale = 1.8 / hypot(*dipoles["S1"])
s0_dipole = request_series(
    "S0 dipole",
    lambda: viewer.remove_vectors("forces"),
    lambda: viewer.set_arrow("dipole", dipoles["S0"], "#2365C6", scale=dipole_scale),
    capture,
)

In [ ]:
print(f'S0 dipole z: {dipoles["S0"][2]:+.3f} a.u.')
display(s0_dipole[-1])
s1_dipole = request_series(
    "S1 dipole",
    lambda: viewer.set_arrow("dipole", dipoles["S1"], "#2365C6", scale=dipole_scale),
    capture,
)

In [ ]:
print(f'S1 dipole z: {dipoles["S1"][2]:+.3f} a.u.')
print("Same dipole scale:", s0_dipole[1]["sets"][0]["scale"] == s1_dipole[0]["sets"][0]["scale"])
display(s1_dipole[-1])

## S₀ → S₁ transition dipole

The two-headed arrow shows the transition dipole axis. Its overall sign is
arbitrary. The oscillator strength for this transition is **f = 0.278**.

In [ ]:
transition = request_series(
    "Transition dipole",
    lambda: viewer.remove_vectors("dipole"),
    lambda: viewer.set_arrow("transition dipole", properties["transitionDipoleAu"]["S0-S1"],
                             "#208375", length=2.4, heads="both"),
    capture,
)

In [ ]:
print(f'f = {properties["oscillatorStrength"]["S0-S1"]:.3f}')
print("Arrow heads:", transition[1]["sets"][0]["heads"])
display(transition[-1])